Import necessary libraries for preprocessing, dataset handling, model training, and evaluation, including `pandas`, `numpy`, `sklearn`, `datasets`, and Hugging Face `transformers`.


In [20]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset
import pandas as pd
import numpy as np
from transformers import pipeline

In [4]:
import transformers
print(transformers.__version__)

4.56.0


Load the processed dataset (`full_data.csv`) containing ~500k entries with text and corresponding emotion labels. Display the first few rows for inspection.


In [5]:

df = pd.read_csv(r'..\data\processed\full_data.csv')
df.head

<bound method NDFrame.head of                                                      text   emotion
0       @tiffanylue i know  i was listenin to bad habi...       sad
1       Layin n bed with a headache  ughhhh...waitin o...       sad
2                     Funeral ceremony...gloomy friday...       sad
3                    wants to hang out with friends SOON!       joy
4       Re-pinging @ghostridah14: why didn't you go to...      fear
...                                                   ...       ...
496217  I was born without a little finger on my right...  surprise
496218                             "Not OJ" license plate  surprise
496219             I welded a rose for my mum's birthday.  surprise
496220  These two different sets of shopping baskets a...  surprise
496221            These removed fish hooks at my local ER  surprise

[496222 rows x 2 columns]>

Encode emotion labels as numeric values using LabelEncoder. Split the dataset into **90% training** and **10% test set** with stratification to preserve label distribution. Convert splits to Hugging Face Dataset objects.


In [6]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["emotion"])


train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])


train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


Tokenize text samples using AutoTokenizer from roberta-base. Apply padding, truncation, and a max length of 128 tokens.


In [7]:
from transformers import AutoTokenizer

model_name = "roberta-base"  
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)


Map: 100%|██████████| 49623/49623 [00:01<00:00, 27238.07 examples/s]


AutoModelForSequenceClassification with 5 output labels corresponding to the target emotions (*Sad, Fear, Anger, Joy, Surprise*). Warn that classifier layers are newly initialized.


In [8]:
from transformers import AutoModelForSequenceClassification

num_labels = 5  # fear, sad, anger, surprise, joy
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Define TrainingArguments for the Hugging Face Trainer, including learning rate, batch size, number of epochs, save strategy, and logging. Initialize the Trainer object with model, tokenizer, datasets, and training arguments.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,  # ajusta según GPU
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)


Fine-tune the RoBERTa model on the training dataset. Monitor training loss, steps per second, and runtime.


In [10]:
trainer.train()


c:\Users\adamc\Documents\GitHub\sp-ml-17-final-project-g1\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,1.314400
200,0.782000
300,0.631200
400,0.554600
500,0.513900
600,0.439100
700,0.456400
800,0.423800
900,0.411200
1000,0.395400


c:\Users\adamc\Documents\GitHub\sp-ml-17-final-project-g1\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\adamc\Documents\GitHub\sp-ml-17-final-project-g1\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=83739, training_loss=0.2145443908530963, metrics={'train_runtime': 291528.28, 'train_samples_per_second': 4.596, 'train_steps_per_second': 0.287, 'total_flos': 6.805580150061338e+16, 'train_loss': 0.2145443908530963, 'epoch': 3.0})

Save the trained model to models/RoBERTa/ for later deployment


In [11]:
trainer.save_model('../models/RoBERTa')

Evaluate the trained model on the test dataset. Compute overall loss, runtime, and per-sample metrics.


In [12]:
trainer.predict(test_dataset)

c:\Users\adamc\Documents\GitHub\sp-ml-17-final-project-g1\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


PredictionOutput(predictions=array([[ 4.029214  , -0.68101203, -1.4632804 ,  0.67985713, -3.2098484 ],
       [ 8.253319  , -1.8081863 , -2.110567  , -1.8238125 , -2.138503  ],
       [-1.9966242 , -2.0934048 , -2.3389645 ,  8.250856  , -2.5802808 ],
       ...,
       [-2.3337152 , -2.3105268 ,  8.587346  , -2.1652496 , -1.9624441 ],
       [-1.9425509 , -2.0426817 , -2.38934   ,  8.274867  , -2.6740575 ],
       [-0.68585783, -1.6205202 ,  4.9573517 , -2.1362534 , -1.5904037 ]],
      shape=(49623, 5), dtype=float32), label_ids=array([1, 0, 3, ..., 2, 3, 2], shape=(49623,)), metrics={'test_loss': 0.24205662310123444, 'test_runtime': 1926.6813, 'test_samples_per_second': 25.756, 'test_steps_per_second': 0.805})

In [13]:
trainer.evaluate()

{'eval_loss': 0.24205662310123444,
 'eval_runtime': 2140.0754,
 'eval_samples_per_second': 23.188,
 'eval_steps_per_second': 0.725,
 'epoch': 3.0}

Generate predictions on the test dataset and compute per-class precision, recall, and F1-score. Overall accuracy: ~92%. Highlight strong performance for Joy and Sad, slightly lower for Surprise.


In [39]:
preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

c:\Users\adamc\Documents\GitHub\sp-ml-17-final-project-g1\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [18]:
print(classification_report(y_true, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

       anger       0.88      0.89      0.89      7636
        fear       0.83      0.86      0.84      6016
         joy       0.95      0.96      0.96     20838
         sad       0.94      0.93      0.94     13229
    surprise       0.81      0.70      0.75      1904

    accuracy                           0.92     49623
   macro avg       0.88      0.87      0.88     49623
weighted avg       0.92      0.92      0.92     49623



Create a Hugging Face pipeline for real-time text classification using the trained model and tokenizer.


Demonstrate model inference on a sample text: “I just want the days to pass until I can see the light in my life again.” Return the predicted emotion and confidence score.


In [ ]:
emotion_classifier = pipeline("text-classification", 
                              model=model, 
                              tokenizer=tokenizer, 
                              return_all_scores=True)

texto = "I just want the days to pass untill I can see the light in my life again"
result = emotion_classifier(texto)

Create a mapping from emotion names to numeric label IDs for easier reference during predictions or visualization.


In [22]:
emotions = np.array(['joy', 'sad', 'anger', 'fear', 'surprise'])
emotions_label = le.transform(emotions)

emotions_dict = {}
for i in range(len(emotions)):
    emotions_dict[f'{emotions[i]}'] = emotions_label[i]

emotions_dict

{'joy': np.int64(2),
 'sad': np.int64(3),
 'anger': np.int64(0),
 'fear': np.int64(1),
 'surprise': np.int64(4)}

Create a mapping from emotion names to numeric label IDs for easier reference during predictions or visualization.


In [27]:
result

[[{'label': 'LABEL_0', 'score': 0.005306741688400507},
  {'label': 'LABEL_1', 'score': 0.020588336512446404},
  {'label': 'LABEL_2', 'score': 0.058160312473773956},
  {'label': 'LABEL_3', 'score': 0.9152349829673767},
  {'label': 'LABEL_4', 'score': 0.0007096245535649359}]]

In [35]:
result[0][3]['score']

0.9152349829673767

In [38]:
def emotion_prediction(text):
    result = emotion_classifier(text)
    for i in range(5):
        if result[0][i]['score'] > 0.85:
            return result[0][i]['label']
    return 'LABEL_5'


emotion_prediction(texto)

'LABEL_3'